# 02b — Integrated inference checkpoint

**Purpose.** Make the expensive stochastic inference explicit and inspectable before Notebook 03 consumes it. The default reuses the distributed replication-level rows only after validating them against the current code, analytical inputs, sample support, checksums, row counts, and deterministic seeds. Set `REGENERATE_INFERENCE = True` to regenerate all 500 pseudo-event assignments and 250 nested-bootstrap replications from the current analytical data.

Stored replication rows are a computational checkpoint, not a final analytical dataset or stored result. Notebook 03 recomputes the reported summaries, confidence intervals, tables, and figures from whichever replication rows are active after this checkpoint.

**Prerequisites.** Run Notebooks 00, 00b, and 01 first. Notebook 02 can also be run before this checkpoint in the public sequence.

**Inputs.** `outputs/data/spotify_song_snapshot_panel.csv`, `outputs/data/master_song_panel.csv`, the current inference code under `src/cdr/`, and, when reusing, the replication rows under `data/inference/`.

**Outputs when regenerating.** `data/inference/dormant_pseudoevent_monte_carlo_replications.csv`, `data/inference/dormant_integrated_bootstrap_replications.csv`, and `data/inference/inference_metadata.json`. Candidate/checkpoint files are retained under `scratch/bootstrap_refit_2026_09_10/`.


In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw_checksums.csv").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display

from cdr import bootstrap_rebuild as br
from cdr import dormant as dm
from cdr import memorydecay as md
from cdr import paths


## Choose reuse or full regeneration

For normal exploration, keep `REGENERATE_INFERENCE = False`. This path fails closed if the stored replications no longer match the current estimator code or analytical inputs.

For a full stochastic rerun, set it to `True`. This can be computationally expensive. The regeneration uses the same primary MemoryDecay estimator inside every bootstrap refit, preserves the recorded deterministic seed strategy, writes the replication-level rows, and validates them before Notebook 03 is run.


In [ ]:
REGENERATE_INFERENCE = False

N_PSEUDOEVENT_REPS = 500
N_BOOTSTRAP_REPS = 250
BOOTSTRAP_JOBS = 1          # Increase only if your notebook environment supports process workers reliably.
BOOTSTRAP_BATCH_SIZE = 10


## Reconstruct the current inference inputs

This step refits the primary snapshot-specific MemoryDecay baselines from the current panel rather than reading rounded fitted parameters. The identity check verifies that a no-resampling refit reproduces the primary estimator exactly.


In [ ]:
try:
    panel, master, wide, first_year_map, fits, source_hashes = br.load_inputs()
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "Missing analytical panels. Run notebooks 00_raw_data_and_billboard, 00b_raw_data_checks, "
        "and 01_linkage_and_analytical_panels first."
    ) from exc

fingerprint = md.dormant_support_fingerprint(panel)

identity = br.identity_refit_check(panel, fits)
display(identity)
assert (identity["max_prediction_difference"] <= 1e-10).all()
assert (identity["primary_n_starts"] == identity["refit_n_starts"]).all()

print("Analytical-support fingerprint:", fingerprint[:16], "...")
print("Current analytical input hashes:", source_hashes)


## Validate or regenerate the replication rows

When reusing, `validate_inference_files` checks the bootstrap-refit version, inference-source hashes, analytical-input hashes, support fingerprint, file checksums, replication counts, consecutive replication IDs, deterministic seed sequences, required columns, finite headline values, and nonempty matched samples. A mismatch stops execution instead of silently falling back to stale inference.

When regenerating, the dedicated rebuild routine creates the full 500/250 replication set from the current analytical objects, preserves a rollback copy of overwritten inference files, publishes the new replication rows, and validates them. It does **not** change the estimand or alter results to reproduce significance.


In [ ]:
if REGENERATE_INFERENCE:
    args = SimpleNamespace(
        assignments=N_PSEUDOEVENT_REPS,
        replications=N_BOOTSTRAP_REPS,
        jobs=BOOTSTRAP_JOBS,
        batch_size=BOOTSTRAP_BATCH_SIZE,
        publish=True,
        accept_reference_update=False,
    )
    result_dir = br.run(args)
    print("Regenerated inference. Checkpoint details:", result_dir)
else:
    metadata = dm.validate_inference_files(fingerprint)
    print("Stored inference is valid for the current code and analytical inputs.")

mc = pd.read_csv(paths.INFERENCE / "dormant_pseudoevent_monte_carlo_replications.csv")
boot = pd.read_csv(paths.INFERENCE / "dormant_integrated_bootstrap_replications.csv")
dm._check_replication_rows(mc, boot)

assert len(mc) == N_PSEUDOEVENT_REPS
assert len(boot) == N_BOOTSTRAP_REPS
print(f"Active inference rows: {len(mc)} pseudo-event assignments; {len(boot)} nested-bootstrap replications")


## Preview the active inference

This is only a diagnostic preview. Notebook 03 remains authoritative for the reported integrated-inference summaries and all downstream tables and figures.


In [ ]:
preview = pd.DataFrame([dm.summarize_stored_inference(mc, boot)]).T
preview.columns = ["value"]
display(preview)
